# 4.1 — Forecasting Performance Across Lead Times

MAE vs lead time for all models at MR=0. Variable-wise performance
with last-value and 24 h persistence baselines.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)
# ── Consistent model ordering (experimental progression) ────────────────────
MODEL_ORDER = ["lstm-baseline-v1", "v32-blind", "v31", "v27", "v30-nll"]
MODEL_ORDER = [r for r in MODEL_ORDER if r in MR0_RUNS]  # keep only available


In [ ]:
# C.aggregate_run("v30-nll", "mr0.00", force=True)

## Figure 4 — Forecasting error (MAE vs lead time)

Per time-of-day window. Models ordered by experimental progression.

In [ ]:
# ── Load extended (per-TOD) caches ───────────────────────────────────────────
EXT = {r: C.load_ext(r, "mr0.00") for r in MODEL_ORDER}

# ── MAE vs lead, by init-hour band — same bands as the σₑ plot (p1b) below ──
# `np.isclose(hod, hr)` matched nothing: forecast-origin times ("blocks"
# index_mode, offset by W/4) never land exactly on the hour — they sit at
# a fixed set of 16 hour-of-day values 1.5h apart (…, 4.34, 5.84, 7.34, …).
# Use the same ±1h band centred on each target hour as p1b / the
# INIT_LO/INIT_HI cell below, so MAE and σₑ are computed on identical
# windows and the notebook doesn't define two different "05 UTC" bins.
INIT_BANDS_MAE = [(4, 6, "05 UTC"), (10, 12, "11 UTC"),
                  (16, 18, "17 UTC"), (22, 24, "23 UTC")]
N_BANDS_MAE = len(INIT_BANDS_MAE)

# ── Compute MAE by init-hour band from raw predictions ──
band_mod_sum = {}
band_mod_cnt = {}
for run in MODEL_ORDER:
    d = C.load_dump(run, "mr0.00")
    P, T, M = d["preds"], d["targets"], d["masks"]
    TH = d["target_hours"].numpy()
    t0h = TH[:, 0]
    hod = t0h % 24

    s_acc = np.zeros((N_BANDS_MAE, K, NV))
    c_acc = np.zeros((N_BANDS_MAE, K, NV))

    BS = 512
    for a0 in range(0, len(P), BS):
        b0 = min(a0 + BS, len(P))
        p = P[a0:b0, :, :, :NV].numpy().astype(np.float64)
        t = T[a0:b0, :, :, :NV].numpy().astype(np.float64)
        m = (M[a0:b0, :, :, :NV].numpy() > 0.5) & KEEP[None, None]
        e_phys = np.abs((p - t) * STD[None, None])

        for bi, (lo_h, hi_h, _) in enumerate(INIT_BANDS_MAE):
            sel = (hod[a0:b0] >= lo_h) & (hod[a0:b0] < hi_h)
            if not sel.any():
                continue
            s_acc[bi] += (e_phys[sel] * m[sel]).sum(axis=(0, 2))
            c_acc[bi] += m[sel].sum(axis=(0, 2))

    band_mod_sum[run] = s_acc
    band_mod_cnt[run] = c_acc
    del d, P, T, M, TH
    import gc; gc.collect()

# ── Persistence MAE baseline per band ──
d = C.load_dump(MODEL_ORDER[0], "mr0.00")
P, T, M = d["preds"], d["targets"], d["masks"]
TH = d["target_hours"].numpy()
t0h = TH[:, 0]
hod = t0h % 24

per_s = np.zeros((N_BANDS_MAE, K, NV))
per_c = np.zeros((N_BANDS_MAE, K, NV))

BS = 512
for a0 in range(0, len(P), BS):
    b0 = min(a0 + BS, len(P))
    t = T[a0:b0, :, :, :NV].numpy().astype(np.float64)
    m = (M[a0:b0, :, :, :NV].numpy() > 0.5) & KEEP[None, None]
    per = t[:, 0:1, :, :].repeat(K, axis=1)
    per_ep = np.abs((per - t) * STD[None, None])

    for bi, (lo_h, hi_h, _) in enumerate(INIT_BANDS_MAE):
        sel = (hod[a0:b0] >= lo_h) & (hod[a0:b0] < hi_h)
        if not sel.any():
            continue
        per_s[bi] += (per_ep[sel] * m[sel]).sum(axis=(0, 2))
        per_c[bi] += m[sel].sum(axis=(0, 2))

del d, P, T, M, TH; gc.collect()

# ── Plot: N_BANDS_MAE rows × NV columns ──
fig, axes = plt.subplots(N_BANDS_MAE, NV, figsize=(17, 13), sharex=True)

for bi, (_, _, band_lbl) in enumerate(INIT_BANDS_MAE):
    for vi, v in enumerate(VARS):
        ax = axes[bi, vi]
        for run in MODEL_ORDER:
            label, col, _ = C.MODELS[run]
            mae = np.where(band_mod_cnt[run] > 0,
                           band_mod_sum[run] / np.maximum(band_mod_cnt[run], 1), np.nan)
            ax.plot(range(1, K), mae[bi, 1:, vi], "o-", ms=2.5, lw=1.2,
                    color=col, label=label)

        per_mae = np.where(per_c > 0, per_s / np.maximum(per_c, 1), np.nan)
        ax.plot(range(1, K), per_mae[bi, 1:, vi], "--", lw=1.1,
                color=C.BASELINE_COLORS["persistence"],
                label=C.BASELINE_LABELS["persistence"])
        ax.grid(alpha=0.3)
        if bi == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if bi == N_BANDS_MAE - 1:
            ax.set_xticks(range(1, K, 2))
            ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    axes[bi, 0].set_ylabel(f"{band_lbl}\nMAE", fontsize=9)

# ── Sync y-axis per variable column ──
for vi in range(NV):
    col_axes = [axes[bi, vi] for bi in range(N_BANDS_MAE)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)

axes[0, -1].legend(fontsize=5.5, loc="upper left")
fig.suptitle("MAE vs lead time, all stations visible (MR=0), by forecast-origin band (±1 h around 05/11/17/23 UTC); dashed = last-value persistence", y=1.01, fontsize=12)
plt.tight_layout()
C.save_fig(fig, "41_mae_vs_lead_by_tod")
plt.show()
plt.close(fig)


In [ ]:
import numpy as np, torch
d = torch.load('../../test_results/v27/best_mr0.00/predictions.pt', map_location='cpu', weights_only=False)
hod = d['target_hours'][:, 0].numpy() % 24
print(f'hod range: {hod.min():.2f} – {hod.max():.2f}')
for h in range(24):
    cnt = ((hod >= h) & (hod < h+1)).sum()
    print(f'  {h:02d}:00 : {cnt}')

## Figure 5 — Error variability (error standard deviation (σₑ) vs lead time)

Per time-of-day window. Same layout as Figure 4.

In [ ]:
# ── Compute 24h-persistence σₑ by TOD (not in extended aggregator) ──────────
# The extended aggregator stores tod_clim_sum/cnt but not sumsq/signed,
# so we compute them here from the same raw data.
#
# NOTE: this used to load a hand-rolled "data/processed/observations.pt" that
# does not exist on disk. The canonical source for raw physical observations
# is common.raw_test_obs() (backed by build_observations(), same as
# aggregate_extended() uses internally) — reuse that instead of a bespoke
# path. raw_test_obs() already returns PHYSICAL-unit values, so no extra
# std scaling is needed here (unlike the model's preds/targets, which are
# normalised). Also fixed CLIM_LAG_STEPS: raw_test_obs() is on the 10-min
# observation grid (6 steps/h), so 24h = 144 steps, not 48 (that value
# assumed a 30-min grid) — use common.py's own CLIM_LAG_STEPS for
# consistency with aggregate_extended().
_run0 = MODEL_ORDER[0]
_d = C.load_dump(_run0, "mr0.00")

raw = C.raw_test_obs()
OBS, OMSK, H0, NT = raw["obs"], raw["mask"], raw["h0"], raw["nt"]

N = len(C.station_table())
KEEP = C.keep_mask(C.station_table(), VARS)

# Accumulators: (4_tod, K, N, V)
clim_sumsq_tod = np.zeros((4, K, N, NV))
clim_signed_tod = np.zeros((4, K, N, NV))
clim_cnt_tod = np.zeros((4, K, N, NV))

TH = _d["target_hours"].numpy()     # (Mw, K)
M = _d["masks"][:, :, :, :NV].numpy() > 0.5  # (Mw, K, N, V)
Mw = TH.shape[0]
CHUNK = 512

for a0 in range(0, Mw, CHUNK):
    b0 = min(a0 + CHUNK, Mw)
    th = TH[a0:b0]       # (chunk, K)
    m = M[a0:b0] & KEEP[None, None]  # (chunk, K, N, V)
    t0h = th[:, 0]        # (chunk,) — same convention as aggregate_extended
    tod = C._tod_of(t0h)  # (chunk,)

    ti_tgt = C.hours_to_row(th, H0)          # (chunk, K)
    ti_clim = ti_tgt - C.CLIM_LAG_STEPS
    ok_t = (ti_tgt >= 0) & (ti_tgt < NT)
    ok_c = (ti_clim >= 0) & (ti_clim < NT)

    truth = OBS[np.clip(ti_tgt, 0, NT-1)].astype(np.float64)   # (chunk, K, N, V)
    tmask = OMSK[np.clip(ti_tgt, 0, NT-1)] & ok_t[:, :, None, None]
    clim_v = OBS[np.clip(ti_clim, 0, NT-1)].astype(np.float64)
    cmask = OMSK[np.clip(ti_clim, 0, NT-1)] & ok_c[:, :, None, None]
    clim_mk = m & tmask & cmask

    clim_err_signed = clim_v - truth   # already physical units (raw_test_obs)
    clim_err_sq = clim_err_signed ** 2

    for ti in range(4):
        s = tod == ti
        if not s.any():
            continue
        clim_sumsq_tod[ti] += (clim_err_sq[s] * clim_mk[s]).sum(0)
        clim_signed_tod[ti] += (clim_err_signed[s] * clim_mk[s]).sum(0)
        clim_cnt_tod[ti] += clim_mk[s].sum(0)

# Pool over stations
clim_sd_tod = C.error_sd(clim_sumsq_tod, clim_signed_tod, clim_cnt_tod)
print("24h-persistence σₑ by TOD computed.")
print(f"  Shape: {clim_sd_tod.shape}")
del OBS, OMSK, _d, raw  # free memory
import gc; gc.collect()


In [ ]:
# ── σₑ vs lead, averaged over init-hour bands (5–7 & 11–13 UTC) ─────────────
# Iterate raw predictions to compute σₑ = sqrt(E[e²] - E[e]²) per band.

INIT_BANDS = [(4, 6, "05 UTC"), (10, 12, "11 UTC"), (16, 18, "17 UTC"), (22, 24, "23 UTC")]
N_BANDS = len(INIT_BANDS)

# ── Compute σₑ by init-hour band from raw predictions ──
band_sumsq  = {}  # {run: (N_BANDS, K, NV)}
band_signed = {}
band_cnt    = {}
for run in MODEL_ORDER:
    d = C.load_dump(run, "mr0.00")
    P, T, M = d["preds"], d["targets"], d["masks"]
    TH = d["target_hours"].numpy()
    t0h = TH[:, 0]
    hod = t0h % 24

    sq_acc  = np.zeros((N_BANDS, K, NV))
    sg_acc  = np.zeros((N_BANDS, K, NV))
    cnt_acc = np.zeros((N_BANDS, K, NV))

    BS = 512
    for a0 in range(0, len(P), BS):
        b0 = min(a0 + BS, len(P))
        p = P[a0:b0, :, :, :NV].numpy().astype(np.float64)
        t = T[a0:b0, :, :, :NV].numpy().astype(np.float64)
        m = (M[a0:b0, :, :, :NV].numpy() > 0.5) & KEEP[None, None]
        e_signed = (p - t) * STD[None, None]
        e_sq = e_signed ** 2

        for bi, (lo_h, hi_h, _) in enumerate(INIT_BANDS):
            sel = (hod[a0:b0] >= lo_h) & (hod[a0:b0] < hi_h)
            if not sel.any():
                continue
            sq_acc[bi]  += (e_sq[sel] * m[sel]).sum(axis=(0, 2))
            sg_acc[bi]  += (e_signed[sel] * m[sel]).sum(axis=(0, 2))
            cnt_acc[bi] += m[sel].sum(axis=(0, 2))

    band_sumsq[run]  = sq_acc
    band_signed[run] = sg_acc
    band_cnt[run]    = cnt_acc
    del d, P, T, M, TH
    import gc; gc.collect()

# ── Persistence baseline σₑ per band ──
d = C.load_dump(MODEL_ORDER[0], "mr0.00")
P, T, M = d["preds"], d["targets"], d["masks"]
TH = d["target_hours"].numpy()
t0h = TH[:, 0]
hod = t0h % 24

per_sq  = np.zeros((N_BANDS, K, NV))
per_sg  = np.zeros((N_BANDS, K, NV))
per_cnt = np.zeros((N_BANDS, K, NV))

BS = 512
for a0 in range(0, len(P), BS):
    b0 = min(a0 + BS, len(P))
    t = T[a0:b0, :, :, :NV].numpy().astype(np.float64)
    m = (M[a0:b0, :, :, :NV].numpy() > 0.5) & KEEP[None, None]
    per = t[:, 0:1, :, :].repeat(K, axis=1)
    e_signed = (per - t) * STD[None, None]
    e_sq = e_signed ** 2

    for bi, (lo_h, hi_h, _) in enumerate(INIT_BANDS):
        sel = (hod[a0:b0] >= lo_h) & (hod[a0:b0] < hi_h)
        if not sel.any():
            continue
        per_sq[bi]  += (e_sq[sel] * m[sel]).sum(axis=(0, 2))
        per_sg[bi]  += (e_signed[sel] * m[sel]).sum(axis=(0, 2))
        per_cnt[bi] += m[sel].sum(axis=(0, 2))



def _sd(sumsq, signed, cnt):
    safe = np.maximum(cnt, 1)
    mean_sq = sumsq / safe
    mean_e  = signed / safe
    var = mean_sq - mean_e ** 2
    return np.where(cnt > 0, np.sqrt(np.maximum(var, 0)), np.nan)

# ── Plot: N_BANDS rows × NV columns ──
fig, axes = plt.subplots(N_BANDS, NV, figsize=(17, 3.5 * N_BANDS), sharex=True)

for bi, (_, _, band_lbl) in enumerate(INIT_BANDS):
    for vi, v in enumerate(VARS):
        ax = axes[bi, vi]
        for run in MODEL_ORDER:
            label, col, _ = C.MODELS[run]
            sd = _sd(band_sumsq[run], band_signed[run], band_cnt[run])
            ax.plot(range(1, K), sd[bi, 1:, vi], "o-", ms=2.5, lw=1.2,
                    color=col, label=label)

        per_sd = _sd(per_sq, per_sg, per_cnt)
        ax.plot(range(1, K), per_sd[bi, 1:, vi], "--", lw=1.1,
                color=C.BASELINE_COLORS["persistence"],
                label=C.BASELINE_LABELS["persistence"])

        ax.grid(alpha=0.3)
        if bi == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if bi == N_BANDS - 1:
            ax.set_xticks(range(1, K, 2))
            ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    axes[bi, 0].set_ylabel(f"{band_lbl}\nσₑ", fontsize=9)

# ── Sync y-axis per variable column ──
for vi in range(NV):
    col_axes = [axes[bi, vi] for bi in range(N_BANDS)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)

axes[0, -1].legend(fontsize=5.5, loc="upper left")
plt.tight_layout()
C.save_fig(fig, "41_error_sd_vs_lead_by_tod")
plt.show()
plt.close(fig)


In [ ]:
# ── MAE at init 05 UTC, +30 min to +6 h ───────────────────────────────────────
# Select windows with T=0 in the same "05 UTC" band as p1/p1b above
# ([4, 6), i.e. ±1h around 05:00) — this used to be the narrower [5, 6),
# a different band definition for the same "05 UTC" label elsewhere in
# the notebook.
INIT_LO, INIT_HI = 4, 6
PLOT_VARS = [v for v in VARS if v != "wind_v"]
NV_PLOT = len(PLOT_VARS)

init5_mae = {}
for run in MODEL_ORDER:
    d = C.load_dump(run, "mr0.00")
    P, T, M = d["preds"], d["targets"], d["masks"]
    TH = d["target_hours"].numpy()
    hod = TH[:, 0] % 24

    s_acc = np.zeros((K, NV))
    c_acc = np.zeros((K, NV))
    BS = 512
    for a0 in range(0, len(P), BS):
        b0 = min(a0 + BS, len(P))
        p = P[a0:b0, :, :, :NV].numpy().astype(np.float64)
        t = T[a0:b0, :, :, :NV].numpy().astype(np.float64)
        m = (M[a0:b0, :, :, :NV].numpy() > 0.5) & KEEP[None, None]
        sel = (hod[a0:b0] >= INIT_LO) & (hod[a0:b0] < INIT_HI)
        if sel.any():
            e = np.abs((p[sel] - t[sel]) * STD[None, None])
            s_acc += (e * m[sel]).sum(axis=(0, 2))
            c_acc += m[sel].sum(axis=(0, 2))
    init5_mae[run] = np.where(c_acc > 0, s_acc / np.maximum(c_acc, 1), np.nan)
    del d, P, T, M, TH
    import gc; gc.collect()

# Persistence baseline
d = C.load_dump(MODEL_ORDER[0], "mr0.00")
P, T, M = d["preds"], d["targets"], d["masks"]
TH = d["target_hours"].numpy()
hod = TH[:, 0] % 24

per_s = np.zeros((K, NV)); per_c = np.zeros((K, NV))
BS = 512
for a0 in range(0, len(P), BS):
    b0 = min(a0 + BS, len(P))
    t = T[a0:b0, :, :, :NV].numpy().astype(np.float64)
    m = (M[a0:b0, :, :, :NV].numpy() > 0.5) & KEEP[None, None]
    sel = (hod[a0:b0] >= INIT_LO) & (hod[a0:b0] < INIT_HI)
    if sel.any():
        per = t[sel, 0:1, :, :].repeat(K, axis=1)
        per_ep = np.abs((per - t[sel]) * STD[None, None])
        per_s += (per_ep * m[sel]).sum(axis=(0, 2))
        per_c += m[sel].sum(axis=(0, 2))

per_mae = np.where(per_c > 0, per_s / np.maximum(per_c, 1), np.nan)
del d, P, T, M, TH; gc.collect()

fig, axes = plt.subplots(1, NV_PLOT, figsize=(15, 3.2), sharex=True)
for vi, v in enumerate(PLOT_VARS):
    ax = axes[vi]
    v_idx = VARS.index(v)
    for run in MODEL_ORDER:
        label, col, _ = C.MODELS[run]
        ax.plot(range(1, K), init5_mae[run][1:, v_idx], "o-", ms=2.5,
                lw=1.2, color=col, label=label)
    ax.plot(range(1, K), per_mae[1:, v_idx], "--", lw=1.1,
            color=C.BASELINE_COLORS["persistence"],
            label=C.BASELINE_LABELS["persistence"])
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xticks(range(1, K, 2))
    ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)

axes[0].set_ylabel("MAE 05 UTC", fontsize=9)
fig.suptitle("MAE vs lead time, forecasts initialised 04–06 UTC, MR=0 (all 155 stations visible), pooled over stations", y=1.04)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", fontsize=7,
           bbox_to_anchor=(1.0, 0.5), frameon=True)
plt.tight_layout(rect=[0, 0, 0.88, 1])
C.save_fig(fig, "41_mae_morning")
plt.show()
plt.close(fig)


## Figure 4b — Forecasting error (MAE vs lead time)

Per season (DJF / MAM / JJA / SON). Models ordered by experimental progression.
Season-split baselines are not available in the base aggregator, so only model
curves are shown.

### MAE and σₑ by season

In [ ]:
# ── Compute σₑ by season from raw predictions ───────────────────────────────
# Extended aggregator lacks season-level sumsq/signed, so we compute from
# raw predictions, one model at a time, to keep memory manageable.
import gc

SEASON_LABELS = ["DJF", "MAM", "JJA", "SON"]
N = len(stn); KEEP = C.keep_mask(stn, VARS)

# Dict: run -> (4_sea, K, V) error SD
sea_sd = {}
sea_mae = {}   # also collect season MAE with baselines
sea_per_sd = None   # persistence σₑ by season
sea_clim_sd = None  # 24h-persistence σₑ by season
sea_per_mae = None
sea_clim_mae = None

raw = C.raw_test_obs()
OBS, OMSK, H0, NT = raw["obs"], raw["mask"], raw["h0"], raw["nt"]

for ri, run in enumerate(MODEL_ORDER):
    d = C.load_dump(run, "mr0.00")
    P = d["preds"][:, :, :, :NV].numpy().astype(np.float64)
    T = d["targets"][:, :, :, :NV].numpy().astype(np.float64)
    M = d["masks"][:, :, :, :NV].numpy() > 0.5
    TH = d["target_hours"].numpy()
    Mw = TH.shape[0]; CHUNK = 512
    del d; gc.collect()

    # Accumulators: (4, K, N, V)
    mod_sumsq = np.zeros((4, K, N, NV))
    mod_signed = np.zeros((4, K, N, NV))
    mod_cnt = np.zeros((4, K, N, NV))
    mod_abssum = np.zeros((4, K, N, NV))  # for MAE
    if ri == 0:  # compute baselines on first model only
        per_sumsq = np.zeros((4, K, N, NV))
        per_signed = np.zeros((4, K, N, NV))
        per_cnt = np.zeros((4, K, N, NV))
        per_abssum = np.zeros((4, K, N, NV))
        clm_sumsq = np.zeros((4, K, N, NV))
        clm_signed = np.zeros((4, K, N, NV))
        clm_cnt = np.zeros((4, K, N, NV))
        clm_abssum = np.zeros((4, K, N, NV))

    for a0 in range(0, Mw, CHUNK):
        b0 = min(a0 + CHUNK, Mw)
        th = TH[a0:b0]
        m = M[a0:b0] & KEEP[None, None]
        p_chunk = P[a0:b0]
        t_chunk = T[a0:b0]
        sea = C._season_of(th[:, 0:1])[:, 0]  # (chunk,) — season from target_hours[:,0]

        mod_err = p_chunk - t_chunk
        mod_abs = np.abs(mod_err)

        # baselines
        if ri == 0:
            ti_tgt = C.hours_to_row(th, H0)
            ti_per = C.hours_to_row(th[:, 0:1], H0)  # persistence: t=0 obs
            ti_clm = ti_tgt - C.CLIM_LAG_STEPS
            ok_t = (ti_tgt >= 0) & (ti_tgt < NT)
            ok_p = (ti_per >= 0) & (ti_per < NT)
            ok_c = (ti_clm >= 0) & (ti_clm < NT)

            truth = OBS[np.clip(ti_tgt, 0, NT-1)].astype(np.float64)
            tmask = OMSK[np.clip(ti_tgt, 0, NT-1)] & ok_t[:, :, None, None]

            per_v = OBS[np.clip(np.broadcast_to(ti_per, th.shape), 0, NT-1)].astype(np.float64)
            pmask = OMSK[np.clip(np.broadcast_to(ti_per, th.shape), 0, NT-1)] & np.broadcast_to(ok_p, th.shape)[:, :, None, None]
            per_mk = m & tmask & pmask
            per_err = per_v - truth

            clm_v = OBS[np.clip(ti_clm, 0, NT-1)].astype(np.float64)
            cmask = OMSK[np.clip(ti_clm, 0, NT-1)] & ok_c[:, :, None, None]
            clm_mk = m & tmask & cmask
            clm_err = clm_v - truth

        for si in range(4):
            s = sea == si
            if not s.any():
                continue
            mod_sumsq[si] += (mod_err[s]**2 * m[s]).sum(0)
            mod_signed[si] += (mod_err[s] * m[s]).sum(0)
            mod_cnt[si] += m[s].sum(0)
            mod_abssum[si] += (mod_abs[s] * m[s]).sum(0)
            if ri == 0:
                per_sumsq[si] += (per_err[s]**2 * per_mk[s]).sum(0)
                per_signed[si] += (per_err[s] * per_mk[s]).sum(0)
                per_cnt[si] += per_mk[s].sum(0)
                per_abssum[si] += (np.abs(per_err[s]) * per_mk[s]).sum(0)
                clm_sumsq[si] += (clm_err[s]**2 * clm_mk[s]).sum(0)
                clm_signed[si] += (clm_err[s] * clm_mk[s]).sum(0)
                clm_cnt[si] += clm_mk[s].sum(0)
                clm_abssum[si] += (np.abs(clm_err[s]) * clm_mk[s]).sum(0)

    sea_sd[run] = C.error_sd(mod_sumsq, mod_signed, mod_cnt)
    sea_mae[run] = C.pool_stations(mod_abssum, mod_cnt)
    if ri == 0:
        sea_per_sd = C.error_sd(per_sumsq, per_signed, per_cnt)
        sea_clim_sd = C.error_sd(clm_sumsq, clm_signed, clm_cnt)
        sea_per_mae = C.pool_stations(per_abssum, per_cnt)
        sea_clim_mae = C.pool_stations(clm_abssum, clm_cnt)
    del P, T, M, TH; gc.collect()
    print(f"  {run}: done")

del OBS, OMSK, raw; gc.collect()
print(f"Season σₑ computed for {len(sea_sd)} models.")


In [ ]:
# ── Compute hourly MAE by season, averaged over +2h30–3h30 leads ─────────────
# Using a range of leads (indices 5,6,7 = +2h30, +3h, +3h30) fills hourly
# bins more evenly and avoids gaps in the diurnal cycle.
import gc, datetime

LEAD_RANGE = [5, 6, 7]  # +2h30, +3h, +3h30
SEASON_LABELS_H = ["DJF", "MAM", "JJA", "SON"]
HOURS = np.arange(24)

raw = C.raw_test_obs()
OBS, OMSK, H0, NT = raw["obs"], raw["mask"], raw["h0"], raw["nt"]
N = len(stn); KEEP_H = C.keep_mask(stn, VARS)

# Results: run -> (4_sea, 24_hour, V)
hourly_mae = {}

for ri, run in enumerate(MODEL_ORDER):
    d = C.load_dump(run, "mr0.00")
    Mw = d["target_hours"].shape[0]

    # Accumulators: (4, 24, N, V)
    mod_sum = np.zeros((4, 24, N, NV))
    mod_cnt = np.zeros((4, 24, N, NV))

    for ki in LEAD_RANGE:
        P = d["preds"][:, ki, :, :NV].numpy().astype(np.float64)
        T = d["targets"][:, ki, :, :NV].numpy().astype(np.float64)
        M = d["masks"][:, ki, :, :NV].numpy() > 0.5
        TH_k = d["target_hours"][:, ki].numpy()
        TH0 = d["target_hours"][:, 0].numpy()

        hod = (TH_k % 24).astype(int)
        ep = datetime.datetime(1970, 1, 1)
        months = np.array([(ep + datetime.timedelta(hours=float(h))).month for h in TH0])
        sea = np.select([np.isin(months, [12,1,2]), np.isin(months, [3,4,5]),
                         np.isin(months, [6,7,8])], [0, 1, 2], 3)

        m = M & KEEP_H[None]
        mod_err = np.abs((P - T) * STD[None])  # physical units (per-station std); axis titles show physical units

        for si in range(4):
            for hi in range(24):
                sel = (sea == si) & (hod == hi)
                if not sel.any():
                    continue
                mod_sum[si, hi] += (mod_err[sel] * m[sel]).sum(0)
                mod_cnt[si, hi] += m[sel].sum(0)

    hourly_mae[run] = C.pool_stations(mod_sum, mod_cnt)
    del mod_sum, mod_cnt; gc.collect()
    print(f"  {run}: done")

del d, OBS, OMSK, raw; gc.collect()
print(f"Hourly MAE by season (+2h30–3h30 avg) computed for {len(hourly_mae)} models.")


In [ ]:
# ── MAE by hour of day and season, +2h30–3h30 average ────────────────────────
fig, axes = plt.subplots(4, NV, figsize=(17, 13), sharex=True)

for si, sea_lbl in enumerate(SEASON_LABELS_H):
    for vi, v in enumerate(VARS):
        ax = axes[si, vi]
        for run in MODEL_ORDER:
            label, col, _ = C.MODELS[run]
            ax.plot(HOURS, hourly_mae[run][si, :, vi], "o-", ms=2.5,
                    lw=1.2, color=col, label=label)
        ax.grid(alpha=0.3)
        if si == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if si == 3:
            ax.set_xticks(range(0, 24, 3))
            ax.set_xlabel("Hour of day (UTC)", fontsize=9)
    axes[si, 0].set_ylabel(f"{sea_lbl}\nMAE", fontsize=9)

# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[si, vi] for si in range(4)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
axes[0, -1].legend(fontsize=5.5, loc="upper left")
fig.suptitle("MAE by target hour of day (UTC) and season, averaged over the +2h30, +3h and +3h30 leads, MR=0, physical units",
             y=1.01, fontsize=12)
plt.tight_layout()
C.save_fig(fig, "41_mae_hourly_by_season")
plt.show()
plt.close(fig)


In [ ]:
# ── Compute hourly σₑ by season, averaged over +2h30–3h30 leads ─────────────
import gc, datetime

raw = C.raw_test_obs()
OBS, OMSK, H0, NT = raw["obs"], raw["mask"], raw["h0"], raw["nt"]
N = len(stn); KEEP_H = C.keep_mask(stn, VARS)

hourly_sd = {}  # run -> (4, 24, V)

for ri, run in enumerate(MODEL_ORDER):
    d = C.load_dump(run, "mr0.00")
    Mw = d["target_hours"].shape[0]

    # Accumulators: (4, 24, N, V)
    mod_sumsq = np.zeros((4, 24, N, NV))
    mod_signed = np.zeros((4, 24, N, NV))
    mod_cnt = np.zeros((4, 24, N, NV))

    for ki in LEAD_RANGE:  # [5, 6, 7]
        P = d["preds"][:, ki, :, :NV].numpy().astype(np.float64)
        T = d["targets"][:, ki, :, :NV].numpy().astype(np.float64)
        M = d["masks"][:, ki, :, :NV].numpy() > 0.5
        TH_k = d["target_hours"][:, ki].numpy()
        TH0 = d["target_hours"][:, 0].numpy()

        hod = (TH_k % 24).astype(int)
        ep = datetime.datetime(1970, 1, 1)
        months = np.array([(ep + datetime.timedelta(hours=float(h))).month for h in TH0])
        sea = np.select([np.isin(months, [12,1,2]), np.isin(months, [3,4,5]),
                         np.isin(months, [6,7,8])], [0, 1, 2], 3)

        m = M & KEEP_H[None]
        err = (P - T) * STD[None]  # signed error, physical units

        for si in range(4):
            for hi in range(24):
                sel = (sea == si) & (hod == hi)
                if not sel.any():
                    continue
                mod_sumsq[si, hi] += (err[sel]**2 * m[sel]).sum(0)
                mod_signed[si, hi] += (err[sel] * m[sel]).sum(0)
                mod_cnt[si, hi] += m[sel].sum(0)

    hourly_sd[run] = C.error_sd(mod_sumsq, mod_signed, mod_cnt)
    del mod_sumsq, mod_signed, mod_cnt; gc.collect()
    print(f"  {run}: done")

del d, OBS, OMSK, raw; gc.collect()
print(f"Hourly σₑ by season (+2h30–3h30 avg) computed for {len(hourly_sd)} models.")


In [ ]:
# ── σₑ by hour of day and season, +2h30–3h30 average ────────────────────────
fig, axes = plt.subplots(4, NV, figsize=(17, 13), sharex=True)

for si, sea_lbl in enumerate(SEASON_LABELS_H):
    for vi, v in enumerate(VARS):
        ax = axes[si, vi]
        for run in MODEL_ORDER:
            label, col, _ = C.MODELS[run]
            ax.plot(HOURS, hourly_sd[run][si, :, vi], "o-", ms=2.5,
                    lw=1.2, color=col, label=label)
        ax.grid(alpha=0.3)
        if si == 0:
            ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if si == 3:
            ax.set_xticks(range(0, 24, 3))
            ax.set_xlabel("Hour of day (UTC)", fontsize=9)
    axes[si, 0].set_ylabel(f"{sea_lbl}\nσₑ", fontsize=9)

# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[si, vi] for si in range(4)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind components ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
lo = min(axes[0, wu_i].get_ylim()[0], axes[0, wv_i].get_ylim()[0])
hi = max(axes[0, wu_i].get_ylim()[1], axes[0, wv_i].get_ylim()[1])
for ri in range(axes.shape[0]):
    axes[ri, wu_i].set_ylim(lo, hi)
    axes[ri, wv_i].set_ylim(lo, hi)
axes[0, -1].legend(fontsize=5.5, loc="upper left")
fig.suptitle("σₑ by hour of day and season — +2h30 to +3h30 average (MR=0.0)",
             y=1.01, fontsize=12)
plt.tight_layout()
C.save_fig(fig, "41_sd_hourly_by_season")
plt.show()
plt.close(fig)


## Interpretation

All models degrade with lead time, but the **Dense** (v31)
consistently achieves the lowest MAE across variables and leads at MR=0.
The LSTM degrades fastest, confirming the value of spatial context.
Pressure decays slowest (high spatial coherence); wind components
decay fastest (turbulent, local).